# SALT Tutorial

SALT evaluates long-form model generations with deterministic ground truth and unit-level correctness labels. This tutorial follows the benchmark flow from motivation and evaluation units, through dataset construction and reproduction data, to vLLM generation and atomic evaluation.

For the full paper context, see [Evaluating LLM Uncertainty in Long-Form Generation Using Deterministic Ground Truth
](SALT_Benchmark_paper.pdf).

In [1]:
# From a fresh environment:
# git clone <repo-url>
# cd SALT
# pip install -e .[dev]
#
# Optional GPU/model generation support:
# pip install -e .[generation]
#
# In Colab, clone or upload the repository first, then uncomment an install line such as:
# !pip install -e .[dev]

## 1. Why SALT?

Large language models are increasingly used for long-form generation: multi-step reasoning, long reports, code, and structured outputs. In these settings, a response is rarely simply right or wrong. Many parts can be correct while a few localized units are mistaken or redundant. A useful uncertainty method should therefore identify which parts are reliable, not only whether the entire answer should be accepted or rejected.

That goal is difficult to evaluate with ordinary long-form benchmarks. Free-form text can have many valid phrasings, factual claims may require external verification, and decomposing a response into atomic claims often depends on fallible human or model-based judges. Label noise is especially damaging for calibration and ranking metrics.

SALT, short for Single-answer Atomic Long-form Target, avoids this by using procedurally generated tasks with a single deterministic long-form ground truth. Each task naturally decomposes into units such as matrix entries, matrix rows, DNA codons, code outputs, logical assignments, or extracted words. This lets us evaluate correctness, redundant units, calibration, and confidence ranking without an external judge.

In [2]:
from __future__ import annotations

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"IProgress not found.*",
    category=Warning,
)

from salt_benchmark.tutorial_utils import configure_quiet_generation_logs

configure_quiet_generation_logs()

from datasets import load_from_disk
from IPython.display import Markdown, display

from salt_benchmark.datasets import (
    create_code_dataset,
    create_dna_dataset,
    create_kronecker_product_dataset,
    create_logic_dataset,
    create_matrix_multiplication_dataset,
    create_needles_in_haystack_dataset,
)
from salt_benchmark.evaluation import (
    auroc_score,
    evaluate_atomic_response,
    evaluate_uncertainty_scores,
    fenced_response_extraction,
    zscore_sigmoid_calibration,
)
from salt_benchmark.generation import CONFIDENCE_SPECS, GenerationConfig, unit_confidence_scores_from_logprobs
from salt_benchmark.tasks import problem_from_task
from salt_benchmark.tutorial_utils import (
    ensure_repository_root,
    redundant_rate,
    run_vllm_generation,
    show_comparisons,
    show_dataset_overview,
    show_metrics_by_granularity,
    show_row,
    show_text,
    show_uncertainty_table,
    unit_precision,
    unit_recall,
)

REPO_ROOT = ensure_repository_root()

## 2. A Matrix Example with Confidence Scores

Consider the following matrix multiplication task. The generated answer can be evaluated at three resolutions:

- `atom`: each matrix element is a separate unit.
- `line`: each row is a separate unit.
- `generation`: the whole extracted answer is one unit.

The fenced response contains two failures. The value `6` should be `5`, and the final value `8` is a redundant unit: an extra matrix entry that violates the ground-truth matrix dimensions. The confidence scores below assign full confidence to the first row and lower confidence to the second row.

In [3]:
matrix_problem = problem_from_task("Matrix Multiplication")

matrix_input = """A=
1 2
3 1
B=
1 2
2 1"""
matrix_reference = """5 4
5 7"""
simulated_generation = """********
5 4
6 7 8
********"""
confidence_matrix = """1.0 1.0
0.8 0.8 0.8"""

extracted_response, start_index, end_index = fenced_response_extraction(simulated_generation)

show_text("Matrix input", matrix_input)
show_text("Ground truth", matrix_reference)
show_text("Simulated LLM response", simulated_generation)
show_text("Confidence scores", confidence_matrix)
show_text("Extracted answer", extracted_response)

toy_evaluations = {
    unit: evaluate_atomic_response(extracted_response, matrix_reference, matrix_problem, unit=unit)
    for unit in ("atom", "line", "generation")
}
toy_confidences = {
    "atom": [1.0, 1.0, 0.8, 0.8, 0.8],
    "line": [1.0, 0.8],
    "generation": [0.88],
}
toy_uncertainty = {
    unit: evaluate_uncertainty_scores(toy_evaluations[unit], scores)
    for unit, scores in toy_confidences.items()
}

show_metrics_by_granularity(toy_evaluations)
show_uncertainty_table(toy_uncertainty)
for unit, evaluation in toy_evaluations.items():
    show_comparisons(f"{unit.title()} comparisons", evaluation)

Matrix input

A=
1 2
3 1
B=
1 2
2 1

Ground truth

5 4
5 7

Simulated LLM response

********
5 4
6 7 8
********

Confidence scores

1.0 1.0
0.8 0.8 0.8

Extracted answer

********
5 4
6 7 8
********

Granularity,Response units,Reference units,Correct,Precision,Recall,Redundant units,Redundant rate
atom,5,4,3,0.600,0.750,1,0.200
line,2,2,1,0.500,0.500,0,0.000
generation,1,1,0,0.000,0.000,0,0.000


Granularity,Precision,ECE,AUROC,PRR
atom,0.600,0.280,0.833,0.667
line,0.500,0.400,1.000,1.000
generation,0.000,0.880,None,nan


#,Response unit,Reference unit,Status
1,5,5,correct
2,4,4,correct
3,6,5,incorrect
4,7,7,correct
5,8,,redundant


#,Response unit,Reference unit,Status
1,5 4,5 4,correct
2,6 7 8,5 7,incorrect


#,Response unit,Reference unit,Status
1,5 46 7 8,5 45 7,incorrect


## 3. Evaluation Metrics

Let $\mathbf{x}=(x_1,\ldots,x_m)$ denote the input sequence and let the model generate $\hat{\mathbf{y}}=(\hat{y}_1,\ldots,\hat{y}_{T_{\mathrm{gen}}})$. SALT evaluates the model's final answer, isolated by explicit output fencing.

SALT decomposes $\hat{\mathbf{y}}$ into evaluation units $\hat{\mathbf{u}}=(\hat{u}_1,\ldots,\hat{u}_{U_{\mathrm{gen}}})$, where each unit is a contiguous token span representing an atomic semantic component. The ground-truth sequence is decomposed analogously into $\mathbf{u}=(u_1,\ldots,u_{U_{\mathrm{gt}}})$. In Matrix Multiplication, an atomic unit is a matrix element and a line unit is a matrix row.

Correctness is defined by strict string equality. A generated unit is correct if and only if it exactly matches the corresponding ground-truth unit. A partial value such as `13` for a ground-truth value `135`, or an incorrect value such as `-135`, receives a binary score of 0.

SALT measures accuracy with precision and recall:

$$
\mathrm{Precision}=\frac{|\mathbf{u}\cap\hat{\mathbf{u}}|}{U_{\mathrm{gen}}},
\qquad
\mathrm{Recall}=\frac{|\mathbf{u}\cap\hat{\mathbf{u}}|}{U_{\mathrm{gt}}}.
$$

Recall reflects the proportion of ground-truth units recovered but ignores hallucinations. Precision considers redundant units and is therefore the paper's main accuracy measure.

A redundant unit is a generated unit that does not appear in the ground-truth sequence. In the matrix task, a redundant unit is an extra element beyond the matrix dimensions, which is mathematically invalid and constitutes a task-specific intrinsic hallucination.

For uncertainty estimation, SALT associates generated units with a confidence score function $\kappa$. For a unit $\hat{u}_i$, this is written as $\kappa(\mathbf{x},\hat{u}_i\mid f)$, and token-level signals can be aggregated to obtain unit-level confidence.

Expected calibration error (ECE) groups instances into confidence bins $B_j$ and compares empirical accuracy with mean confidence:

$$
\mathrm{ECE}=\sum_{j=1}^{m}\frac{|B_j|}{N}\left|\mathrm{acc}(B_j)-\mathrm{conf}(B_j)\right|.
$$

Confidence ranking asks whether a confidence function assigns higher scores to correct units than to incorrect ones. Under 0/1 loss, AUROC is equivalent to this ranking probability:

$$
\Pr\Big[
\kappa(\mathbf{x}^{(1)}\hat{\mathbf{u}}^{(1)}_{1:i-1},\hat{u}^{(1)}_i)
<
\kappa(\mathbf{x}^{(2)}\hat{\mathbf{u}}^{(2)}_{1:j-1},\hat{u}^{(2)}_j)
\;\Big|\;
\hat{u}^{(1)}_i\neq u^{(1)}_i\land \hat{u}^{(2)}_j=u^{(2)}_j
\Big].
$$

Prediction Rejection Ratio (PRR) compares the rejection curve induced by a confidence score with random rejection and oracle rejection. Higher PRR means the confidence ordering closes more of the gap between random rejection and the best possible rejection order.

In [4]:
toy_atom = toy_evaluations["atom"]

show_text("Response used for atom metrics", toy_atom.clean_response)
show_text("Reference used for atom metrics", matrix_reference)
show_comparisons("Atom-level units used for these metrics", toy_atom)

print(f"Atom precision: {toy_atom.metrics['num_correct']} / {toy_atom.metrics['num_response_units']} = {unit_precision(toy_atom):.3f}")
print(f"Atom recall: {toy_atom.metrics['num_correct']} / {toy_atom.metrics['num_reference_units']} = {unit_recall(toy_atom):.3f}")
print(f"Atom slot coverage: {toy_atom.metrics['slot_coverage']:.3f}")
print(f"Redundant atom count: {toy_atom.metrics['num_redundant']}")
print(f"Redundant atom rate: {redundant_rate(toy_atom):.3f}")

Response used for atom metrics

5 4
6 7 8

Reference used for atom metrics

5 4
5 7

#,Response unit,Reference unit,Status
1,5,5,correct
2,4,4,correct
3,6,5,incorrect
4,7,7,correct
5,8,,redundant


Atom precision: 3 / 5 = 0.600
Atom recall: 3 / 4 = 0.750
Atom slot coverage: 1.000
Redundant atom count: 1
Redundant atom rate: 0.200


### Confidence Direction And Calibration

`mean_probability`, `median_probability`, `max_probability`, and `log_probability_sum` increase with confidence. `perplexity`, `log_perplexity`, `mean_entropy`, and `max_entropy` increase with uncertainty, so they must be inverted for ranking metrics such as AUROC and PRR.

ECE should use calibrated confidence values in `[0, 1]`. The paper normalizes raw scores with a calibration set, using several calibration method. A simple example of calibration methodology uses z-scoring followed by a sigmoid, and then applies that transform to held-out scores.

In [5]:
perplexity_scores = [1.0, 1.0, 1.8, 1.8, 1.8]
perplexity_spec = CONFIDENCE_SPECS["perplexity"]
calibrated_perplexity = zscore_sigmoid_calibration(
    perplexity_scores,
    perplexity_scores,
    higher_is_confident=perplexity_spec["higher_is_confident"],
)
perplexity_uncertainty = evaluate_uncertainty_scores(
    toy_evaluations["atom"],
    perplexity_scores,
    score_name="perplexity",
    calibration_confidences=calibrated_perplexity,
)

generated_atom_comparisons = [comparison for comparison in toy_evaluations["atom"].comparisons if comparison.response_unit is not None]
score_rows = [
    "| Atom | Correct | Perplexity | Calibrated Confidence |",
    "|---|---:|---:|---:|",
]
for comparison, perplexity, calibrated in zip(generated_atom_comparisons, perplexity_scores, calibrated_perplexity):
    score_rows.append(
        f"| `{comparison.response_unit}` | `{int(comparison.is_correct)}` | `{perplexity:.3f}` | `{calibrated:.3f}` |"
    )

display(Markdown("\n".join(score_rows)))
show_uncertainty_table({"perplexity": perplexity_uncertainty})

| Atom | Correct | Perplexity | Calibrated Confidence |
|---|---:|---:|---:|
| `5` | `1` | `1.000` | `0.773` |
| `4` | `1` | `1.000` | `0.773` |
| `6` | `0` | `1.800` | `0.307` |
| `7` | `1` | `1.800` | `0.307` |
| `8` | `0` | `1.800` | `0.307` |

Granularity,Precision,ECE,AUROC,PRR
perplexity,0.600,0.107,0.833,0.667


## 4. Constructing New SALT Data

SALT datasets are procedural. Each task exposes a small creator function that samples an input and computes the exact reference answer. You can fix a seed for reproducible examples, change the seed to produce a different benchmark split, or pass `seed=None` for non-deterministic generation.

The generated rows use a shared structure: `task`, `input`, and `reference`, with task-specific columns added when needed.

In [6]:
new_salt_datasets = {
    "DNA Translation": create_dna_dataset(num_samples=3, min_codons=4, max_codons=6, seed=42),
    "First-Order Logic": create_logic_dataset(num_samples=3, num_variables_range=(4, 6), seed=42),
    "Matrix Multiplication": create_matrix_multiplication_dataset(
        num_samples=3,
        output_lengths=[16],
        min_inner_dim=4,
        max_inner_dim=4,
    ),
    "Kronecker Product": create_kronecker_product_dataset(num_samples=3, output_lengths=[9]),
    "Code": create_code_dataset(num_samples=3),
    "Words Collection": create_needles_in_haystack_dataset(num_samples=3),
}

show_dataset_overview(new_salt_datasets)
show_row("Fresh matrix multiplication sample", new_salt_datasets["Matrix Multiplication"][0])
show_row("Fresh DNA Translation sample", new_salt_datasets["DNA Translation"][0])

Dataset,Rows,Columns
DNA Translation,3,"task, input, reference"
First-Order Logic,3,"task, input, reference"
Matrix Multiplication,3,"task, input, reference"
Kronecker Product,3,"task, input, reference"
Code,3,"task, variant, input, reference"
Words Collection,3,"task, input, subject, reference"


Fresh matrix multiplication sample

Input

A=
5 3 -1 -1
7 -8 4 -6
B=
-8 1 9 4 5 4 5 0
-7 6 -1 0 -2 -6 8 5
3 -2 6 1 -1 -1 -5 -8
1 7 -8 7 6 -4 3 -6

Reference

-65 18 44 12 14 7 51 29
6 -91 143 -10 11 96 -67 -36

Fresh DNA Translation sample

Input

AAT CCC AAG AAA CCA CGC

Reference

TTA GGG TTC TTT GGT GCG

## 5. Loading the Paper Data

The repository also includes the ready-to-use reproduction datasets used for the paper experiments. These are stored with HuggingFace `Dataset.save_to_disk`, so they can be loaded directly from `data/reproduction`.

In [7]:
reproduction_paths = {
    "DNA Translation": "data/reproduction/DNA_dataset",
    "First-Order Logic": "data/reproduction/Logic_dataset",
    "Matrix Multiplication": "data/reproduction/Matrix_multiplication_dataset",
    "Kronecker Product": "data/reproduction/Kronecker_product_dataset",
    "Code": "data/reproduction/Code_dataset",
    "Words Collection": "data/reproduction/NeedlesInHaystack_dataset",
}

paper_datasets = {
    name: load_from_disk(str(REPO_ROOT / relative_path))
    for name, relative_path in reproduction_paths.items()
}

show_dataset_overview(paper_datasets)
show_row("Paper First-Order Logic sample", paper_datasets["First-Order Logic"][0])

Dataset,Rows,Columns
DNA Translation,300,"task, input, reference"
First-Order Logic,400,"task, input, reference"
Matrix Multiplication,300,"task, input, reference"
Kronecker Product,300,"task, input, reference"
Code,148,"task, input, reference"
Words Collection,300,"task, input, reference, subject, subject_set"


Paper First-Order Logic sample

Input

∧(p4, p6)
¬(↔(p3, ∨(p2, p3)))
¬(→(↔(p1, p2), p3))
∧(p2, p5, →(p6, p2))
↔(p5, ∨(p4, p6))
↔(p5, ∧(p5, p6))


Initial Assignments:
p4: True


Reference

p1: True
p2: True
p3: False
p4: True
p5: True
p6: True

### From Dataset Rows To Problem Objects

Rows store the task name as a string. Use `problem_from_task` to instantiate the matching problem class before evaluating a response for that row.

In [8]:
row = paper_datasets["Matrix Multiplication"][0]
row_problem = problem_from_task(row)
row_response = f"********\n{row['reference']}\n********"
row_result = evaluate_atomic_response(row_response, row["reference"], row_problem, unit="atom")

print(row_problem.name)
print(f"precision={row_result.metrics['precision']:.3f}, recall={row_result.metrics['recall']:.3f}")

Matrix Multiplication
precision=1.000, recall=1.000


### Minimal Batch Evaluation

For aggregate metrics, loop over dataset rows and model responses, instantiate each row's problem, evaluate the fenced response, and collect the metrics you need.

In [9]:
batch_rows = list(paper_datasets["DNA Translation"].select(range(3)))
batch_responses = [f"********\n{row['reference']}\n********" for row in batch_rows]

batch_metrics = []
for row, response in zip(batch_rows, batch_responses):
    problem = problem_from_task(row)
    evaluation = evaluate_atomic_response(response, row["reference"], problem, unit="atom")
    batch_metrics.append(
        {
            "task": problem.name,
            "precision": evaluation.metrics["precision"],
            "recall": evaluation.metrics["recall"],
            "redundant": evaluation.metrics["num_redundant"],
        }
    )

batch_metrics

[{'task': 'DNA Translation', 'precision': 1.0, 'recall': 1.0, 'redundant': 0},
 {'task': 'DNA Translation', 'precision': 1.0, 'recall': 1.0, 'redundant': 0},
 {'task': 'DNA Translation', 'precision': 1.0, 'recall': 1.0, 'redundant': 0}]

## 6. Preparing a Real Prompt

Each task object owns the benchmark prompt format: the task instruction, the fenced-answer requirement, a few demonstrations, and the test input.

The prompt below is built from a generated Matrix Multiplication row and can be sent directly to vLLM.

In [10]:
generation_problem = problem_from_task("Matrix Multiplication")
generation_row = new_salt_datasets["Matrix Multiplication"][0]
generation_prompt = generation_problem.construct_prompt(generation_row["input"], few_shot_num=2)

show_text("Prompt sent to the model", generation_prompt, max_chars=3000)
show_text("Reference answer", generation_row["reference"])

Prompt sent to the model

You are given two input matrices: `A` and `B`. Every matrix element is provided row by row, separated by spaces. Matrix rows are separated by new lines. You have to calculate the matrix multiplication of `A` and `B` and print the matrix solution in the same format.
You can write anything you want to help yourself, but right before you write your answer, you should print '********' in a single line. At the end of your answer please place another '********' to signal the answer is finished. MAKE SURE YOU FENCE YOUR FINAL ANSWER WITH '********' AT THE BEGINNING AND THE END, OR ELSE, YOUR ANSWER WILL AUTOMATICALLY FAIL!
Here are 2 examples:
Example 1:
A=
1 2
3 4
B=
5 6
7 8
Answer:
********
19 22
43 50
********
Example 2:
A=
-1 0 2
B=
3
4
5
Answer:
********
7
********
Please answer the following as demonstrated before:
A=
5 3 -1 -1
7 -8 4 -6
B=
-8 1 9 4 5 4 5 0
-7 6 -1 0 -2 -6 8 5
3 -2 6 1 -1 -1 -5 -8
1 7 -8 7 6 -4 3 -6
Solution:

Reference answer

-65 18 44 12 14 7 51 29
6 -91 143 -10 11 96 -67 -36

## 7. Running Real Generation With vLLM

Set `RUN_REAL_VLLM` to control model execution in this section. When it is `True`, the cell sends the prompt to vLLM and prints the raw generation. When it is `False`, the notebook skips model execution while preserving the surrounding evaluation flow.

The request uses 20 top logprobs to match the OpenAI and Gemini API limit.

In [11]:
RUN_REAL_VLLM = True
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

real_generation = None
if RUN_REAL_VLLM:
    real_generation = run_vllm_generation(
        MODEL_NAME,
        generation_prompt,
        generation_row["input"],
        GenerationConfig(max_tokens=2048, temperature=1.0, top_p=1.0, logprobs=20),
    )
    show_text("Raw model generation", real_generation.text, max_chars=4000)
else:
    display(Markdown("`RUN_REAL_VLLM` is `False`; no model generation was run."))

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Raw model generation

We are given two matrices A and B, and we need to compute their matrix multiplication A × B.

---

**Step 1: Parse Matrix A**

A =
5 3 -1 -1
7 -8 4 -6

This is a 2×4 matrix.

**Step 2: Parse Matrix B**

B =
-8 1 9 4 5 4 5 0
-7 6 -1 0 -2 -6 8 5
3 -2 6 1 -1 -1 -5 -8
1 7 -8 7 6 -4 3 -6

This is a 4×8 matrix.

---

**Step 3: Matrix Multiplication**

Since A is 2×4 and B is 4×8, the product A×B will be a 2×8 matrix.

To compute (A×B)[i][j] = sum over k from 0 to 3 of A[i][k] × B[k][j]

Let’s compute each element.

---

**Compute row 0 of A: [5, 3, -1, -1]**

- Element (0,0): 5*(-8) + 3*(-7) + (-1)*3 + (-1)*1 = -40 -21 -3 -1 = -65  
- Element (0,1): 5*1 + 3*6 + (-1)*(-2) + (-1)*7 = 5 + 18 + 2 -7 = 18  
- Element (0,2): 5*9 + 3*(-1) + (-1)*6 + (-1)*(-8) = 45 -3 -6 +8 = 44  
- Element (0,3): 5*4 + 3*0 + (-1)*1 + (-1)*7 = 20 + 0 -1 -7 = 12  
- Element (0,4): 5*5 + 3*(-2) + (-1)*(-1) + (-1)*6 = 25 -6 +1 -6 = 14  
- Element (0,5): 5*4 + 3*(-6) + (-1)*(-1) + (-1)*(-4) = 20 -18 +1 +4 = 7  
- Element (0,6): 5*5 + 3*8 + (-1)*(-5) + (-1)*3 = 25 +24 +5 -3 = 51  
- Element (0,7): 5*0 + 3*5 + (-1)*(-8) + (-1)*(-6) = 0 +15 +8 +6 = 29  

---

**Compute row 1 of A: [7, -8, 4, -6]**

- Element (1,0): 7*(-8) + (-8)*(-7) + 4*3 + (-6)*1 = -56 + 56 + 12 -6 = 6  
- Element (1,1): 7*1 + (-8)*6 + 4*(-2) + (-6)*7 = 7 -48 -8 -42 = -81  
- Element (1,2): 7*9 + (-8)*(-1) + 4*6 + (-6)*(-8) = 63 +8 +24 +48 = 143  
- Element (1,3): 7*4 + (-8)*0 + 4*1 + (-6)*7 = 28 + 0 +4 -42 = -10  
- Element (1,4): 7*5 + (-8)*(-2) + 4*(-1) + (-6)*6 = 35 +16 -4 -36 = 11  
- Element (1,5): 7*4 + (-8)*(-6) + 4*(-1) + (-6)*(-4) = 28 +48 -4 +24 = 96  
- Element (1,6): 7*5 + (-8)*8 + 4*(-5) + (-6)*3 = 35 -64 -20 -18 = -67  
- Element (1,7): 7*0 + (-8)*5 + 4*(-8) + (-6)*(-6) = 0 -40 -32 +36 = -36  

---

**Final Result Matrix (2×8):**

Row 0: -65 18 44 12 14 7 51 29  
Row 1: 6 -81 143 -10 11 96 -67 -36

---

********
-65 18 44 12 14 7 51 29
6 -81 143 -10 11 96 -67 -36
********

## 8. Evaluating the Real Generation

SALT evaluation extracts the fenced final answer, decomposes it into units, and compares each generated unit with the deterministic reference.

When model output is available, the cell below prints the extracted answer and reports atom, line, and generation metrics.

In [12]:
if real_generation is None:
    display(Markdown("No vLLM generation is available for evaluation."))
else:
    real_extracted_response, real_start, real_end = fenced_response_extraction(real_generation.text)
    show_text("Extracted model answer", real_extracted_response or "<no fenced answer found>", max_chars=4000)

    if real_extracted_response:
        real_evaluations = {
            unit: evaluate_atomic_response(
                real_extracted_response,
                generation_row["reference"],
                generation_problem,
                unit=unit,
            )
            for unit in ("atom", "line", "generation")
        }
        show_metrics_by_granularity(real_evaluations)
        for unit, evaluation in real_evaluations.items():
            show_comparisons(f"Real {unit} comparisons", evaluation, limit=16)

Extracted model answer

********
-65 18 44 12 14 7 51 29
6 -81 143 -10 11 96 -67 -36
********

Granularity,Response units,Reference units,Correct,Precision,Recall,Redundant units,Redundant rate
atom,16,16,15,0.938,0.938,0,0.000
line,2,2,1,0.500,0.500,0,0.000
generation,1,1,0,0.000,0.000,0,0.000


#,Response unit,Reference unit,Status
1,-65,-65,correct
2,18,18,correct
3,44,44,correct
4,12,12,correct
5,14,14,correct
6,7,7,correct
7,51,51,correct
8,29,29,correct
9,6,6,correct
10,-81,-91,incorrect


#,Response unit,Reference unit,Status
1,-65 18 44 12 14 7 51 29,-65 18 44 12 14 7 51 29,correct
2,6 -81 143 -10 11 96 -67 -36,6 -91 143 -10 11 96 -67 -36,incorrect


#,Response unit,Reference unit,Status
1,-65 18 44 12 14 7 51 296 -81 143 -10 11 96 -67 -36,-65 18 44 12 14 7 51 296 -91 143 -10 11 96 -67 -36,incorrect


## 9. Uncertainty Estimation Evaluation

When vLLM returns top-logprobs, SALT maps token-level scores back to the extracted answer units and evaluates calibration, ranking, and rejection quality. The table also repeats Precision, because uncertainty trends do not necessarily follow the accuracy trend.

In [13]:
if real_generation is None or not real_generation.logprobs:
    display(Markdown("No vLLM logprobs are available for uncertainty evaluation."))
elif "real_evaluations" not in globals():
    display(Markdown("Run the real-generation evaluation step before uncertainty evaluation."))
else:
    real_uncertainty = {}
    skipped_units = []
    for unit, evaluation in real_evaluations.items():
        confidences = unit_confidence_scores_from_logprobs(
            real_generation.text,
            real_generation.logprobs,
            generation_problem,
            unit=unit,
        )
        if len(confidences) != evaluation.metrics["num_response_units"]:
            skipped_units.append(unit)
            continue
        real_uncertainty[unit] = evaluate_uncertainty_scores(evaluation, confidences)

    if real_uncertainty:
        show_uncertainty_table(real_uncertainty)
    if skipped_units:
        display(Markdown("Logprobs could not be aligned for: " + ", ".join(f"`{unit}`" for unit in skipped_units) + "."))

Granularity,Precision,ECE,AUROC,PRR
atom,0.938,0.062,0.733,0.467
line,0.500,0.500,1.000,1.000
generation,0.000,1.000,None,nan


## 10. Advanced Notes

### Adding Your Own Tasks

A SALT task is a `Problem` object plus rows of deterministic data. The class gives SALT the task name, a solver or reference function, a prompt, optional few-shot examples, and the atom-decomposition metadata used by evaluation and confidence alignment.

The key atomization fields are `delimiter` and `regex_exp`. If `delimiter` is set, atom evaluation splits each non-empty line on that literal delimiter. If `delimiter=None`, SALT falls back to `regex_exp` and every regex match becomes one atom. Use a delimiter for simple translations and flat sequences. Use a regex when punctuation or nesting is part of the semantic unit, such as matrices, lists of varying length, nested lists, and so on.

Existing tasks show the tradeoff. `MatrixMultiplicationProblem` uses a space delimiter, so atom units are scalar entries and line units are rows. Some Code variants also use delimiters: `ReverseWordsProblem` splits words with spaces, while `NextPermutationProblem` splits list elements with commas. The more structured Code variants need regexes: `RotateProblem` extracts full matrix rows, and `InsertProblem` extracts full intervals. Those regexes have to be precise because splitting on commas would turn `[1, 5]` into two atoms instead of one interval.

The example below creates a new task named `Decimal-Hexa-Translation`. It mirrors DNA Translation: every input token has exactly one output token, the solver is deterministic, and a space delimiter is enough because each decimal number maps to one hexadecimal value.

In [14]:
from datasets import Dataset
from IPython.display import Markdown, display

from salt_benchmark.evaluation import decompose_to_units, evaluate_atomic_response
from salt_benchmark.tasks import (
    EditDistanceProblem,
    InsertProblem,
    MatrixMultiplicationProblem,
    NextPermutationProblem,
    ReverseWordsProblem,
    RotateProblem,
)
from salt_benchmark.tasks.base import FENCE_INSTRUCTION, Problem
from salt_benchmark.tutorial_utils import show_comparisons, show_dataset_overview, show_row, show_text


def atomization_summary(problem, response):
    return {
        "delimiter": repr(problem.delimiter),
        "uses_regex": problem.regex_exp is not None and problem.delimiter is None,
        "atoms": decompose_to_units(response, problem, unit="atom"),
        "lines": decompose_to_units(response, problem, unit="line"),
    }


atomization_examples = [
    (
        "Matrix Multiplication",
        MatrixMultiplicationProblem(),
        "5 4\n5 7",
        "space-delimited scalar entries; line units remain matrix rows",
    ),
    (
        "Reverse Words",
        ReverseWordsProblem(),
        "olleH dlroW",
        "space-delimited words",
    ),
    (
        "Next Permutation",
        NextPermutationProblem(),
        "[1, 3, 2]",
        "comma-delimited list elements",
    ),
    (
        "Rotate Matrix",
        RotateProblem(),
        "[[3, 1], [-4, -2]]",
        "regex extracts whole matrix rows, not individual numbers",
    ),
    (
        "Insert Interval",
        InsertProblem(),
        "[[1, 5], [6, 9]]",
        "regex extracts whole intervals, including decimal endpoints",
    ),
]

rows = [
    "| Task | Delimiter | Regex Used For Atoms? | Example Atom Units | Why This Boundary? |",
    "|---|---:|---:|---|---|",
]
for task_name, problem, response, note in atomization_examples:
    summary = atomization_summary(problem, response)
    rows.append(
        f"| {task_name} | `{summary['delimiter']}` | {summary['uses_regex']} | "
        f"`{summary['atoms']}` | {note} |"
    )

display(Markdown("\n".join(rows)))


def decimal_to_hexa_sequence(decimal_sequence: str, delimiter: str = " ") -> str:
    values = []
    for token in decimal_sequence.split(delimiter):
        if not token.strip():
            continue
        value = int(token)
        if value < 0:
            raise ValueError("Decimal-Hexa-Translation expects non-negative decimal integers.")
        values.append(format(value, "X"))
    return delimiter.join(values)


DECIMAL_HEXA_FEW_SHOT = [
    ("10 15 16 31", "A F 10 1F"),
    ("0 1 8 255", "0 1 8 FF"),
]


class DecimalHexaTranslationProblem(Problem):
    def __init__(self, few_shot_examples: list[tuple[str, str]] | None = None):
        super().__init__(
            name="Decimal-Hexa-Translation",
            delimiter=" ",
            regex_exp=None,
            few_shot_examples=few_shot_examples or DECIMAL_HEXA_FEW_SHOT.copy(),
            solution=decimal_to_hexa_sequence,
        )

    def construct_prompt(self, test_input: str, few_shot_num: int = 8) -> str:
        few_shot_prompt = self.construct_few_shot_prompt(few_shot_num)
        return (
            "You are given a sequence of non-negative decimal integers. Translate each decimal integer "
            "to its uppercase hexadecimal value and preserve the same space-delimited order.\n"
            f"{FENCE_INSTRUCTION}\n"
            f"{few_shot_prompt}"
            f"Please answer the following as demonstrated before:\nDecimal sequence = {test_input}\nAnswer ="
        )


DECIMAL_HEXA_INPUTS = [
    "0 1 10 15",
    "16 31 32 255",
    "2 3 4 5 6",
    "8 9 10 11 12",
    "26 42 100 4095",
]


def create_decimal_hexa_translation_dataset() -> Dataset:
    references = [decimal_to_hexa_sequence(sample_input) for sample_input in DECIMAL_HEXA_INPUTS]
    return Dataset.from_dict(
        {
            "task": ["Decimal-Hexa-Translation"] * len(DECIMAL_HEXA_INPUTS),
            "input": DECIMAL_HEXA_INPUTS,
            "reference": references,
        }
    )


decimal_hexa_problem = DecimalHexaTranslationProblem()
decimal_hexa_dataset = create_decimal_hexa_translation_dataset()
decimal_hexa_sample = decimal_hexa_dataset[0]
decimal_hexa_response = f"********\n{decimal_hexa_sample['reference']}\n********"
decimal_hexa_evaluation = evaluate_atomic_response(
    decimal_hexa_response,
    decimal_hexa_sample["reference"],
    decimal_hexa_problem,
    unit="atom",
)

show_dataset_overview({"Decimal-Hexa-Translation": decimal_hexa_dataset})
show_row("Decimal-Hexa mini-dataset sample", decimal_hexa_sample)
show_text("Decimal-Hexa prompt", decimal_hexa_problem.construct_prompt(decimal_hexa_sample["input"], few_shot_num=2))
show_comparisons("Decimal-Hexa atom comparisons", decimal_hexa_evaluation)

| Task | Delimiter | Regex Used For Atoms? | Example Atom Units | Why This Boundary? |
|---|---:|---:|---|---|
| Matrix Multiplication | `' '` | False | `['5', '4', '5', '7']` | space-delimited scalar entries; line units remain matrix rows |
| Reverse Words | `' '` | False | `['olleH', 'dlroW']` | space-delimited words |
| Next Permutation | `','` | False | `['1', '3', '2']` | comma-delimited list elements |
| Rotate Matrix | `None` | True | `['[3, 1]', '[-4, -2]']` | regex extracts whole matrix rows, not individual numbers |
| Insert Interval | `None` | True | `['[1, 5]', '[6, 9]']` | regex extracts whole intervals, including decimal endpoints |

Dataset,Rows,Columns
Decimal-Hexa-Translation,5,"task, input, reference"


Decimal-Hexa mini-dataset sample

Input

0 1 10 15

Reference

0 1 A F

Decimal-Hexa prompt

You are given a sequence of non-negative decimal integers. Translate each decimal integer to its uppercase hexadecimal value and preserve the same space-delimited order.
You can write anything you want to help yourself, but right before you write your answer, you should print '********' in a single line. At the end of your answer please place another '********' to signal the answer is finished. MAKE SURE YOU FENCE YOUR FINAL ANSWER WITH '********' AT THE BEGINNING AND THE END, OR ELSE, YOUR ANSWER WILL AUTOMATICALLY FAIL!
Here are 2 examples:
Example 1:
10 15 16 31
Answer:
********
A F 10 1F
********
Example 2:
0 1 8 255
Answer:
********
0 1 8 FF
********
Please answer the following as demonstrated before:
Decimal sequence = 0 1 10 15
Answer =

#,Response unit,Reference unit,Status
1,0,0,correct
2,1,1,correct
3,A,A,correct
4,F,F,correct


### Macro-AUROC And Micro-AUROC

Macro-AUROC pools generated units across examples and ranks them together. It answers whether the confidence score separates correct from incorrect units globally. Micro-AUROC ranks units within each example first and then aggregates, so each example contributes its own ranking problem. The next cell uses two fixed Words Collection examples from the paper's Multi-Needle task and derives correctness labels from SALT's evaluator rather than writing the labels by hand.

In [15]:
needles_problem = problem_from_task("Words Collection")
needles_examples = [
    {
        "example": "colors",
        "text": "the red ribbon crossed the blue gate before green paint dried",
        "subject": "colors",
        "response": "********\nred yellow green\n********",
        "confidences": [0.70, 0.65, 0.68],
    },
    {
        "example": "weekdays",
        "text": "monday plans waited near friday and sunday closed the report",
        "subject": "weekdays",
        "response": "********\nmonday thursday sunday\n********",
        "confidences": [0.50, 0.45, 0.48],
    },
]

for example in needles_examples:
    reference = needles_problem.solution(example["text"], example["subject"])
    evaluation = evaluate_atomic_response(example["response"], reference, needles_problem, unit="atom")
    generated_comparisons = [comparison for comparison in evaluation.comparisons if comparison.response_unit is not None]
    example["reference"] = reference
    example["response_units"] = [comparison.response_unit for comparison in generated_comparisons]
    example["correctness"] = [1 if comparison.is_correct else 0 for comparison in generated_comparisons]
    example["auroc"] = auroc_score(example["confidences"], example["correctness"])
    assert len(example["confidences"]) == len(example["correctness"])

macro_confidences = [score for example in needles_examples for score in example["confidences"]]
macro_correctness = [label for example in needles_examples for label in example["correctness"]]
macro_auroc = auroc_score(macro_confidences, macro_correctness)
micro_aurocs = [example["auroc"] for example in needles_examples]
micro_auroc = sum(score for score in micro_aurocs if score is not None) / len(micro_aurocs)

assert macro_auroc is not None

example_rows = [
    "| Example | Subject | Reference | Response Units | Correctness From Evaluator | Confidence Scores | Per-Example AUROC |",
    "|---|---|---|---|---:|---:|---:|",
]
for example in needles_examples:
    example_rows.append(
        f"| {example['example']} | `{example['subject']}` | `{example['reference']}` | "
        f"`{example['response_units']}` | `{example['correctness']}` | "
        f"`{example['confidences']}` | `{example['auroc']:.2f}` |"
    )

metric_rows = [
    "| Metric | Ranking Scope | Correctness Labels | Confidence Scores | AUROC |",
    "|---|---|---:|---:|---:|",
    f"| Macro-AUROC | all generated Words Collection units pooled | `{macro_correctness}` | `{macro_confidences}` | `{macro_auroc:.2f}` |",
    f"| Micro-AUROC | average of the two per-example rankings | `{[example['auroc'] for example in needles_examples]}` | per example | `{micro_auroc:.2f}` |",
]

show_text("Words Collection example 1 text", needles_examples[0]["text"])
show_text("Words Collection example 2 text", needles_examples[1]["text"])
display(Markdown("\n".join(example_rows) + "\n\n" + "\n".join(metric_rows)))

Words Collection example 1 text

the red ribbon crossed the blue gate before green paint dried

Words Collection example 2 text

monday plans waited near friday and sunday closed the report

| Example | Subject | Reference | Response Units | Correctness From Evaluator | Confidence Scores | Per-Example AUROC |
|---|---|---|---|---:|---:|---:|
| colors | `colors` | `red blue green` | `['red', 'yellow', 'green']` | `[1, 0, 1]` | `[0.7, 0.65, 0.68]` | `1.00` |
| weekdays | `weekdays` | `monday friday sunday` | `['monday', 'thursday', 'sunday']` | `[1, 0, 1]` | `[0.5, 0.45, 0.48]` | `1.00` |

| Metric | Ranking Scope | Correctness Labels | Confidence Scores | AUROC |
|---|---|---:|---:|---:|
| Macro-AUROC | all generated Words Collection units pooled | `[1, 0, 1, 1, 0, 1]` | `[0.7, 0.65, 0.68, 0.5, 0.45, 0.48]` | `0.75` |
| Micro-AUROC | average of the two per-example rankings | `[1.0, 1.0]` | per example | `1.00` |

### Strict Indexing And Dynamic Alignment On DNA Translation

The example below uses the DNA Translation task: produce the complementary codon sequence for a DNA input. The response omits the first reference codon, so strict positional indexing compares every later codon against the wrong slot. Passing `dynamic_alignment=True` uses a simplified Needleman-Wunsch global alignment over response and reference units, inserts a gap for the omitted codon, and then resynchronizes the remaining codons. Because that gap occurs before later generated units, it is still a precision mistake: alignment can recover the later codons, but it does not make the omission free.

In [16]:
import html


def colored_unit(unit, is_correct):
    text = "missing" if unit is None else str(unit)
    color = "#137333" if is_correct else "#b3261e"
    background = "#e6f4ea" if is_correct else "#fce8e6"
    return (
        f"<span style='display:inline-block; min-width:4.5rem; padding:0.15rem 0.4rem; "
        f"border-radius:0.35rem; color:{color}; background:{background}; font-weight:600;'>"
        f"{html.escape(text)}</span>"
    )


def precision_summary(evaluation):
    precision_units = evaluation.metrics.get("num_precision_units", evaluation.metrics["num_response_units"])
    return f"{evaluation.metrics['num_correct']} / {precision_units} = {evaluation.metrics['precision']:.3f}"


def display_alignment_table(title, strict_evaluation, dynamic_evaluation):
    rows = []
    for strict_comparison, dynamic_comparison in zip(strict_evaluation.comparisons, dynamic_evaluation.comparisons):
        rows.append(
            "<tr>"
            f"<td>{colored_unit(strict_comparison.response_unit, strict_comparison.is_correct)}</td>"
            f"<td>{colored_unit(dynamic_comparison.response_unit, dynamic_comparison.is_correct)}</td>"
            f"<td><code>{html.escape(str(dynamic_comparison.reference_unit))}</code></td>"
            "</tr>"
        )

    rows.append(
        "<tr style='border-top:2px solid #d0d7de;'>"
        f"<td><strong>Precision: {precision_summary(strict_evaluation)}</strong></td>"
        f"<td><strong>Precision: {precision_summary(dynamic_evaluation)}</strong></td>"
        "<td><strong>Generated plus internal-missing units</strong></td>"
        "</tr>"
    )

    display(
        {
            "text/html": f"""
<h3>{html.escape(title)}</h3>
<p><strong>DNA input:</strong> <code>{html.escape(dna_input)}</code></p>
<table>
<thead>
<tr><th>Response Unit</th><th>Aligned Response Unit</th><th>Reference Unit</th></tr>
</thead>
<tbody>{''.join(rows)}</tbody>
</table>
""",
            "text/plain": title,
        },
        raw=True,
    )


dna_problem = problem_from_task("DNA Translation")
dna_input = "AGA GCT AAG GCC"
dna_reference = "TCT CGA TTC CGG"
dna_shifted_response = """********
CGA TTC CGG
********"""

strict_dna_evaluation = evaluate_atomic_response(dna_shifted_response, dna_reference, dna_problem, unit="atom")
dynamic_dna_evaluation = evaluate_atomic_response(
    dna_shifted_response,
    dna_reference,
    dna_problem,
    unit="atom",
    dynamic_alignment=True,
)

display_alignment_table("Strict vs dynamic alignment for DNA Translation", strict_dna_evaluation, dynamic_dna_evaluation)

Response Unit,Aligned Response Unit,Reference Unit
CGA,missing,TCT
TTC,CGA,CGA
CGG,TTC,TTC
missing,CGG,CGG
Precision: 0 / 3 = 0.000,Precision: 3 / 4 = 0.750,Generated plus internal-missing units
